In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv(r'spam.csv', encoding='latin-1')

In [ ]:
df.sample(5)

In [ ]:
df.shape

In [ ]:
# 1. Data cleaning
# 2. EDA
# 3. Text Preprocessing
# 4. Model building
# 5. Evaluation
# 6. Improvement
# 7. Website
# 8. Deploy

## 1. Data Cleaning

In [ ]:
df.info()

In [ ]:
# drop last 3 cols
df.drop(columns=['Unnamed: 2','Unnamed: 3','Unnamed: 4'],inplace=True)

In [ ]:
df.sample(5)

In [ ]:
# renaming the cols
df.rename(columns={'v1':'target','v2':'text'},inplace=True)
df.sample(5)

In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

In [ ]:
df['target'] = encoder.fit_transform(df['target'])

In [ ]:
df.head()

In [ ]:
# missing values
df.isnull().sum()

In [ ]:
# check for duplicate values
df.duplicated().sum()

In [ ]:
# remove duplicates
df = df.drop_duplicates(keep='first')

In [ ]:
df.duplicated().sum()

In [ ]:
df.shape

## 2.EDA

In [ ]:
df.head()

In [ ]:
df['target'].value_counts()

In [ ]:
import matplotlib.pyplot as plt
plt.pie(df['target'].value_counts(), labels=['ham','spam'],autopct="%0.2f")
plt.show()

In [ ]:
# Data is imbalanced

In [ ]:
import nltk

In [ ]:
!pip install nltk

In [ ]:
nltk.download('punkt')

In [ ]:
df['num_characters'] = df['text'].apply(len)

In [ ]:
df.head()

In [ ]:
import nltk
nltk.download('punkt_tab')

In [ ]:
# num of words
df['num_words'] = df['text'].apply(lambda x:len(nltk.word_tokenize(x)))

In [ ]:
df.head()

In [ ]:
df['num_sentences'] = df['text'].apply(lambda x:len(nltk.sent_tokenize(x)))

In [ ]:
df.head()

In [ ]:
df[['num_characters','num_words','num_sentences']].describe()

In [ ]:
# ham
df[df['target'] == 0][['num_characters','num_words','num_sentences']].describe()

In [ ]:
#spam
df[df['target'] == 1][['num_characters','num_words','num_sentences']].describe()

In [ ]:
import seaborn as sns

In [ ]:
plt.figure(figsize=(12,6))
sns.histplot(df[df['target'] == 0]['num_characters'])
sns.histplot(df[df['target'] == 1]['num_characters'],color='red')

In [ ]:
plt.figure(figsize=(12,6))
sns.histplot(df[df['target'] == 0]['num_words'])
sns.histplot(df[df['target'] == 1]['num_words'],color='red')

In [ ]:
sns.pairplot(df,hue='target')

In [ ]:
sns.heatmap(df.corr(numeric_only=True), annot=True)

## 3. Data Preprocessing
- Lower case
- removing html tags, urls
- emoji handling
- Tokenization
- Removing special characters
- Removing stop words and punctuation
- Stemming

In [ ]:
# import re
# def remove_tags(raw_text):
#     cleaned_text = re.sub(re.compile('<.*?>'), '', raw_text)
#     return cleaned_text

In [ ]:
# df['cleaned_text'] = df['text'].apply(remove_tags)

In [ ]:
# def remove_url(raw_text):
#     cleaned_text = re.sub(re.compile(r'https?://\S+|www\. \S+'),'',raw_text)
#     return cleaned_text

In [ ]:
# df['cleaned_text'] = df['cleaned_text'].apply(remove_url)

In [ ]:
# df['cleaned_text'] = df['cleaned_text'].apply(lambda x:x.lower())

In [ ]:
# from nltk.corpus import stopwords
# import nltk
# nltk.download('stopwords')
# sw_list = stopwords.words('english')

# df['cleaned_text'] = df['cleaned_text'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))

In [ ]:
# df['cleaned_text']

## 4. Model Building


# BERT Transformer Fine-Tuning
Why DistilBERT?
- Faster than BERT
- Smaller model size
- Excellent performance on text classification
- Works well for spam/email/SMS classification


In [ ]:

# !pip install transformers datasets accelerate evaluate -q


In [ ]:

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from datasets import Dataset
import evaluate


In [ ]:

# Use cleaned dataframe
bert_df = df[['text', 'target']].copy()

bert_df = bert_df.dropna()

train_texts, val_texts, train_labels, val_labels = train_test_split(
    bert_df['text'],
    bert_df['target'],
    test_size=0.2,
    random_state=42,
    stratify=bert_df['target']
)

print("Train Shape:", train_texts.shape)
print("Validation Shape:", val_texts.shape)


In [ ]:

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(texts):

    return tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_encodings = tokenize_function(train_texts.tolist())
val_encodings = tokenize_function(val_texts.tolist())


In [ ]:

train_dataset = Dataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'labels': train_labels.tolist()
})

val_dataset = Dataset.from_dict({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask'],
    'labels': val_labels.tolist()
})

print(train_dataset)
print(val_dataset)


In [ ]:

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)


In [ ]:

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )

    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="weighted"
    )

    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"]
    }


In [ ]:

training_args = TrainingArguments(
    output_dir="./bert_results",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_dir="./logs",

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    compute_metrics=compute_metrics
)


In [ ]:

trainer.train()


In [ ]:

predictions = trainer.predict(val_dataset)

preds = np.argmax(predictions.predictions, axis=1)

print(classification_report(val_labels, preds))


In [ ]:

model.save_pretrained("bert_sms_classifier")
tokenizer.save_pretrained("bert_sms_classifier")

print("BERT model saved successfully!")


In [ ]:

# Test on custom message

sample_text = "Congratulations! You won a free iPhone. Click now!"

inputs = tokenizer(
    sample_text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)

with torch.no_grad():

    outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1)

label_map = {
    0: "Not Spam",
    1: "Spam"
}

print("Prediction:", label_map[prediction.item()])
